# AMMS 302 — Week 11: Health Care Data Standards
**HL7 FHIR R4 · OMOP CDM · ICD-10 · สปสช. Data Dictionary**

> เปิดคู่กับ [สไลด์ wk11](./wk11.html) — Lab: parse FHIR bundle + ICD-10 mapping + สร้าง OMOP-mini จากข้อมูลเรา

### 🎯 Learning objectives (CLO1/CLO4)
- อธิบายที่มา/บทบาทของ FHIR, OMOP, ICD-10 ได้
- อ่าน FHIR Bundle JSON และคลี่เป็นตารางด้วย pandas ได้
- Map รหัส ICD-10 → chapter/group และสร้าง mini OMOP tables ได้

### 📚 Official references
- **FHIR R4:** [hl7.org/fhir](https://www.hl7.org/fhir/) · [Patient](https://www.hl7.org/fhir/patient.html) · [Observation](https://www.hl7.org/fhir/observation.html) · [Bundle](https://www.hl7.org/fhir/bundle.html) · [JSON format](https://www.hl7.org/fhir/json.html)
- **OMOP CDM v5.4:** [CommonDataModel](https://ohdsi.github.io/CommonDataModel/cdm54.html) · [person](https://ohdsi.github.io/CommonDataModel/cdm54.html#person) · [Book of OHDSI](https://ohdsi.github.io/TheBookOfOhdsi/) · [ATLAS demo](https://atlas-demo.ohdsi.org/)
- **ICD-10:** [WHO ICD-10 browse](https://icd.who.int/browse10/2019/en)
- Python: [json_normalize](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)

---


In [ ]:
# Setup — ไฟล์ตัวอย่าง FHIR bundle ในโฟลเดอร์นี้
import json, pathlib, sqlite3
import pandas as pd
assert pathlib.Path("patient_fhir_demo.json").exists(), "patient_fhir_demo.json not found"
print("ready")

## §1 FHIR Bundle — anatomy (สไลด์ 05–06)
Bundle = container · `entry[]` แต่ละชิ้นมี `resource` · resourceType = Patient/Observation/… · fields เป็น camelCase ตาม spec


In [ ]:
# §1.1 โหลด + สำรวจโครงสร้าง
with open("patient_fhir_demo.json", encoding="utf-8") as f:
    bundle = json.load(f)
print("resourceType:", bundle["resourceType"], "| id:", bundle["id"])
print("entries:", len(bundle.get("entry", [])))
first = bundle["entry"][0]["resource"]
print(json.dumps(first, ensure_ascii=False, indent=2)[:400])

# §1.2 นับ resource types
types = pd.Series([e["resource"]["resourceType"] for e in bundle.get("entry", [])])
print(types.value_counts())

## §2 json_normalize → flat table (สไลด์ 06)


In [ ]:
# §2 flatten entries
df_fhir = pd.json_normalize(bundle, record_path=["entry"])
display(df_fhir.head())
print(list(df_fhir.columns))

# §2.1 สกัด field สำคัญ (dot paths ตาม FHIR spec)
patients = pd.DataFrame({
  "fhir_id":  df_fhir["resource.id"],
  "gender":   df_fhir["resource.gender"],
  "birthDate":df_fhir["resource.birthDate"],
})
patients["age"] = pd.to_datetime("today").year - pd.to_datetime(patients["birthDate"]).dt.year
display(patients)

## §3 ICD-10 — โครงสร้าง & mapping (สไลด์ 04)
ICD-10 code = `Chapter letter + digits` เช่น `E11.9` = Type 2 DM — เรา map หัว chapter ด้วย dict

🔗 browse: https://icd.who.int/browse10/2019/en


In [ ]:
# §3 ICD-10 chapter map (ย่อ)
icd_chapters = {
 'A':'I Infectious','B':'I Infectious','C':'II Neoplasms','D':'II Neoplasms/Blood',
 'E':'III Endocrine (E11=DM)','F':'IV Mental','G':'V Nervous',
 'I':'IX Circulatory','J':'X Respiratory','K':'XI Digestive','N':'XIV Genitourinary',
 'O':'XV Pregnancy','S':'XVII Injury','T':'XVII Injury/Poison','Z':'XXI Factors'}
samples = ['E11.9','I48','I10','J45.909','K35.80']
icd_df = pd.DataFrame({'code':samples,
                       'chapter':[icd_chapters[c[0]] for c in samples]})
display(icd_df)

# §3.1 ใช้กับ DB เรา: เพิ่มคอลัมน์ icd จำลองใน patients แล้ว map
con = sqlite3.connect("healthinfo.db") if pathlib.Path('healthinfo.db').exists() else None
if con:
    demo = pd.read_sql("SELECT patient_id, hba1c FROM patients LIMIT 8", con)
    demo['dx_icd'] = demo['hba1c'].apply(lambda a: 'E11.9' if pd.notna(a) and a>=6.5 else 'Z00.0')
    demo['chapter']= demo['dx_icd'].str[0].map(icd_chapters)
    display(demo)

## §4 Mini-OMOP — แปลงข้อมูลเราเป็น CDM style (สไลด์ 07–08)
OMOP หลัก: `person`(person_id PK), `visit_occurrence`, `condition_occurrence`, `observation` — ทุกตารางผูก person_id


In [ ]:
# §4 สร้าง mini CDM จาก patients_data.csv → SQLite omop_mini.db
import sqlite3, pathlib
raw = pd.read_csv("patients_data.csv")
omop_path = pathlib.Path("omop_mini.db")
if omop_path.exists(): omop_path.unlink()
oc = sqlite3.connect(omop_path); oc.execute("PRAGMA foreign_keys=ON"); ocur=oc.cursor()
ocur.execute("CREATE TABLE person(person_id INTEGER PRIMARY KEY, gender_concept TEXT, year_of_birth INT)")
ocur.execute("CREATE TABLE condition_occurrence(condition_occurrence_id INTEGER PRIMARY KEY,
             person_id INTEGER REFERENCES person(person_id), condition_concept_name TEXT)")
for _,r in raw.iterrows():
    try: pid=int(float(r['subject_id']))
    except: continue
    yob=None
    try: yob=int(float(str(r['admission_date'])[:4]))-int(float(r['Age'])) if r['Age']==r['Age'] else None
    except: pass
    g='MALE' if r['gender']=='ชาย' else ('FEMALE' if r['gender']=='หญิง' else None)
    ocur.execute("INSERT OR IGNORE INTO person VALUES(?,?,?)",(pid,g,yob))
    dx='Diabetes (E11)' if pd.notna(r['HbA1c_level']) and r['HbA1c_level']>=6.5 else 'Screening (Z00)'
    ocur.execute("INSERT INTO condition_occurrence VALUES(?,?,?)",(pid,pid,dx))
oc.commit()
display(pd.read_sql("SELECT * FROM person LIMIT 5", oc))
display(pd.read_sql("""
 SELECT c.condition_concept_name AS dx, COUNT(*) n,
        AVG(p.year_of_birth) avg_yob
 FROM condition_occurrence c JOIN person p USING(person_id)
 GROUP BY dx ORDER BY n DESC""", oc))
oc.close(); print("✅ omop_mini.db created — schema ตาม OMOP naming")

### ✅ Self-check
- นับ resourceTypes ได้ Patient=3
- json_normalize columns มี `resource.id/gender/birthDate`
- E11→Endocrine, I→Circulatory mapping ถูก
- omop_mini.db มี person + condition_occurrence (FK enforced)

### 📝 Homework 11
1) เพิ่ม `visit_occurrence(person_id FK, visit_date)` ≥5 แถว + query join person  
2) อ่าน [FHIR Patient page](https://www.hl7.org/fhir/patient.html) แล้ว list 5 fields ที่ dataset เราไม่มี (email/address…)  
3) ส่ง .ipynb + omop_mini.db

---
### 🔗 Specs
[FHIR](https://www.hl7.org/fhir/) · [OMOP CDM](https://ohdsi.github.io/CommonDataModel/cdm54.html) · [ICD-10](https://icd.who.int/browse10/2019/en) · [Book of OHDSI](https://ohdsi.github.io/TheBookOfOhdsi/)
